# SN-03 — Sirenisation, phase 2 : top 5 par commune

Matching probabiliste sur les PM non validees en phase 1, avec blocking
**communal** : une personne morale n'est comparee qu'aux unites legales de sa
propre commune. C'est la maille d'origine du projet ; la phase 3 elargira au
departement.

Le scoring est vectorise : meme formule, calculee en matrices plutot qu'en
boucles. L'equivalence est stricte — ecart nul sur les trois scores, memes
candidats, meme ordre — et verifiee par `tests/test_equivalence_vectorisation.py`.

Le bonus de +15 pour un SIREN coherent est conserve, ainsi que sa desactivation
quand le SIREN a deja ete valide en phase 1 par une PM jumelle.

La lecture du parquet reste departementale, seule maille de partitionnement :
chaque departement est lu une fois puis decoupe en communes en memoire.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
from src.sirenisation import CONFIG_SN, COLS_UL_EXPORT
from src.phases import executer_phase, estimer_charge
from src.excel_export import export_topn_excel, LABELS
from src.display import afficher_tableau, afficher_synthese
from config.settings import (
    SN_PERIMETRE, SIRENE_UL_CLEAN, SIRENE_UL_PARTITIONNE,
    SN_VALIDES_P1, SN_PHASE2, SN_VALIDES_P2, RESULTS_SN_DIR, BUDGET_MEMOIRE_GO,
)

RESULTS_SN_DIR.mkdir(parents=True, exist_ok=True)
VALIDES = {"VALIDE_FORT", "VALIDE"}

sirens_valides_p1 = set(
    pd.read_parquet(SN_VALIDES_P1)["siren_norm_pm"].dropna().astype(str))
sirens_valides_p1.discard("")
print(f"SIREN valides en P1 : {len(sirens_valides_p1):,}")

df_perimetre = pd.read_parquet(SN_PERIMETRE)
print(f"PM a traiter : {len(df_perimetre):,}")
print(f"   {df_perimetre['sous_ensemble'].value_counts().to_dict()}")

SIREN valides en P1 : 38,618
PM a traiter : 17,408
   {'A': 12637, 'B': 4154, 'C': 617}


In [2]:
tailles = (pd.read_parquet(SIRENE_UL_CLEAN, columns=["code_commune_norm_ul"])
             .groupby("code_commune_norm_ul").size())
estimation = estimer_charge(df_perimetre, CONFIG_SN, tailles,
                            niveau_matching="commune")
display(estimation.head(15))


Maille          : commune
Paires a scorer : 216,394,418
Blocs           : 6447


,nb_structures,nb_candidats,paires,lot_max
06088,102.0,120999.0,12341898.0,344
31555,90.0,137064.0,12335760.0,303
75108,54.0,179277.0,9680958.0,232
33063,89.0,101398.0,9024422.0,410
75116,75.0,103502.0,7762650.0,402
34172,81.0,89995.0,7289595.0,462
75117,73.0,97262.0,7100126.0,428
75115,67.0,88654.0,5939818.0,469
44109,70.0,80190.0,5613300.0,519
59350,67.0,68308.0,4576636.0,609


In [3]:
df_top5 = executer_phase(
    df_perimetre, CONFIG_SN,
    chemin_partitionne=SIRENE_UL_PARTITIONNE,
    chemin_complet=SIRENE_UL_CLEAN,
    n=5, approfondi=False,
    identifiants_deja_valides=sirens_valides_p1,
    colonnes_sirene=COLS_UL_EXPORT,
    budget_go=BUDGET_MEMOIRE_GO,
    niveau_matching="commune",
)

df_top5 = df_top5.rename(columns={"siren": "siren_ref",
                                  "nom_sirene_retenu": "nom_ul_retenu",
                                  "identifiant_coherent": "siren_coherent",
                                  "score_ajuste": "score_global_ajuste"})

rang1 = df_top5[df_top5["rang"] == 1]
print(f"\nLignes top 5 : {len(df_top5):,}")
print(f"SIREN coherent au rang 1 : {int(rang1['siren_coherent'].sum()):,}")
print(f"Bonus applique           : {int(df_top5['bonus_applique'].sum()):,}")
print()
print(rang1["statut_candidat"].value_counts())

Phase 2 (commune):   0%|          | 0/103 [00:00<?, ?it/s]

/home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/src/phases.py:139: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(resultats + orphelins, ignore_index=True) if (resultats or orphelins) \



Lignes top 5 : 87,012
SIREN coherent au rang 1 : 4,766
Bonus applique           : 6,074

statut_candidat
VALIDE           11091
DOUTEUX           3359
VALIDE_FORT       2841
REJETE             110
SANS_CANDIDAT        7
Name: count, dtype: int64


In [4]:
afficher_tableau(
    df_top5[["NB_PmSmsseId", "TX_DenominationPm", "siren_ref",
             "denominationUniteLegale", "nom_ul_retenu",
             "score_nom", "score_adresse", "score_global",
             "score_global_ajuste", "statut_candidat", "rang"]],
    "Apercu top 5 phase 2", max_lignes=10)

NB_PmSmsseId,TX_DenominationPm,siren_ref,denominationUniteLegale,nom_ul_retenu,score_nom,score_adresse,score_global,score_global_ajuste,statut_candidat,rang
6,CLINIQUE DU SOUFFLE LE PONTET,503330615,STYL'EPIS,STYL EPIS,25.000000,100.000000,70.000000,70.000000,VALIDE,1
6,CLINIQUE DU SOUFFLE LE PONTET,989829460,L'INSTITUT CECILE SALON DE BEAUTE,INSTITUT CECILE SALON BEAUTE,39.800000,78.260000,62.870000,62.870000,VALIDE,2
6,CLINIQUE DU SOUFFLE LE PONTET,485168082,CADAULAN,CADAULAN,29.290000,70.000000,53.720000,53.720000,VALIDE,3
6,CLINIQUE DU SOUFFLE LE PONTET,984748210,LAGERTHA TAXI,LAGERTHA TAXI,8.000000,75.080000,48.250000,48.250000,DOUTEUX,4
6,CLINIQUE DU SOUFFLE LE PONTET,799069265,CHARPENTES DU PLATEAU,CHARPENTES PLATEAU,46.050000,49.050000,47.850000,47.850000,DOUTEUX,5
25,EHPAD ST TRIVIER DE COURTES,260100243,MAISON DE RETRAITE ST TRIVIER DE COURTES,MAISON RETRAITE ST TRIVIER COURTES,64.740000,38.280000,48.860000,63.860000,DOUTEUX,1
25,EHPAD ST TRIVIER DE COURTES,646750125,SOCIETE TRAVAUX FACON ST AMOUR,SOCIETE TRAVAUX FACON ST AMOUR,35.440000,70.000000,56.180000,56.180000,VALIDE,2
25,EHPAD ST TRIVIER DE COURTES,106097322,L'EVEIL DES COQ'AIN,EVEIL COQ AIN,34.530000,70.000000,55.810000,55.810000,VALIDE,3
25,EHPAD ST TRIVIER DE COURTES,753804319,T.F.C.,T F C,32.790000,70.000000,55.120000,55.120000,VALIDE,4
25,EHPAD ST TRIVIER DE COURTES,981198369,PHARMACIE DE SAINT TRIVIER DE COURTES,PHARMACIE SAINT TRIVIER COURTES,65.490000,47.780000,54.860000,54.860000,DOUTEUX,5


In [5]:
COLS_EXPORT = [
    "ID_PmSmsse", "NB_PmSmsseId", "TX_NumFinessPm", "TX_Siren",
    "TX_CategorieEntiteGeographiqueExercice", "TX_DenominationPm",
    "cdcommune_norm_pm", "adresse_complete_pm", "sous_ensemble",
    "siren_ref", "siren_coherent", "bonus_applique",
    "denominationUniteLegale", "sigleUniteLegale", "nom_ul_retenu",
    "adresse_siege_complete_ul", "codeCommuneEtablissement",
    "categorieJuridiqueUniteLegale", "activitePrincipaleUniteLegale",
    "dateCreationUniteLegale",
    "score_nom", "score_adresse", "score_global", "score_global_ajuste",
    "statut_candidat", "rang",
]

df_top5["rang"] = df_top5["rang"].astype(int)
compteurs = export_topn_excel(df_top5, SN_PHASE2, COLS_EXPORT, sheet_name="Top5")

afficher_synthese({LABELS[s]: n for s, n in compteurs.items()},
                  "Synthese phase 2 sirenisation (rang 1)")
print(f"\nFichier : {SN_PHASE2}")

Statut,Nb,% du total
Valide_fort,"2,841",16.3%
Valide,"11,091",63.7%
Douteux,"3,359",19.3%
Rejeté,110,0.6%
Sans_candidat,7,0.0%
TOTAL,"17,408",100.0%



Fichier : /home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/results/sirenisation/sirenisation_phase2_top5.xlsx


In [6]:
valides = rang1[rang1["statut_candidat"].isin(VALIDES)][
    ["NB_PmSmsseId", "siren_ref", "statut_candidat"]].copy()
valides.to_parquet(SN_VALIDES_P2, index=False)
print(f"{len(valides):,} PM validees en phase 2 -> {SN_VALIDES_P2}")

13,932 PM validees en phase 2 -> /home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/data/processed/sirenisation_valides_p2.parquet
